# Experiment 5: Naïve Bayesian Classifier

## Overview
The Naïve Bayesian classifier is a probabilistic classifier based on Bayes' theorem with the assumption of conditional independence between features. Despite its simplicity, it performs well in many real-world applications.

## Key Concepts
- **Bayes' Theorem**: P(A|B) = P(B|A) * P(A) / P(B)
- **Naïve Assumption**: All features are conditionally independent given the class
- **Likelihood**: P(features|class)
- **Prior Probability**: P(class)
- **Posterior Probability**: P(class|features)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.datasets import load_breast_cancer
import matplotlib.pyplot as plt
import seaborn as sns

print("Naïve Bayesian Classifier")
print("=" * 60)

## Dataset

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names

# Create dataframe for visualization
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("\nBreast Cancer Dataset:")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Class names: {target_names}")
print(f"\nFirst few rows:")
print(df.head())

## Data Splitting

In [ ]:
# Split data: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Further split test into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nClass distribution in training set: {np.bincount(y_train)}")
print(f"Class distribution in test set: {np.bincount(y_test)}")

## Train Naïve Bayesian Classifier

In [ ]:
# Train Naïve Bayesian classifier
nb_classifier = GaussianNB()
nb_classifier.fit(X_train, y_train)

print("\nNaïve Bayes Classifier trained!")
print(f"\nClass priors (P(class)):")
for class_idx, class_name in enumerate(target_names):
    prior = np.mean(y_train == class_idx)
    print(f"  P({class_name}): {prior:.4f}")

## Predictions on Different Sets

In [ ]:
# Make predictions
y_train_pred = nb_classifier.predict(X_train)
y_val_pred = nb_classifier.predict(X_val)
y_test_pred = nb_classifier.predict(X_test)

# Calculate accuracies
train_accuracy = accuracy_score(y_train, y_train_pred)
val_accuracy = accuracy_score(y_val, y_val_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("\n" + "="*60)
print("ACCURACY SCORES")
print("="*60)
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Performance Metrics

In [ ]:
# Calculate metrics
precision = precision_score(y_test, y_test_pred, average='weighted')
recall = recall_score(y_test, y_test_pred, average='weighted')
f1 = f1_score(y_test, y_test_pred, average='weighted')

print("\n" + "="*60)
print("PERFORMANCE METRICS (Test Set)")
print("="*60)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=target_names))

## Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Confusion Matrix - Naïve Bayesian Classifier')
plt.tight_layout()
plt.show()

# Print confusion matrix analysis
print("\nConfusion Matrix Analysis:")
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")
print(f"\nSpecificity: {tn/(tn+fp):.4f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")

## Prediction Probabilities

In [ ]:
# Get prediction probabilities
test_proba = nb_classifier.predict_proba(X_test)

print("\nPrediction Probabilities (First 10 test samples):")
for i in range(min(10, len(X_test))):
    pred = y_test_pred[i]
    true = y_test[i]
    confidence = test_proba[i, pred]
    print(f"\nSample {i+1}:")
    print(f"  True Class: {target_names[true]}")
    print(f"  Predicted: {target_names[pred]} (confidence: {confidence:.4f})")
    for class_idx, class_name in enumerate(target_names):
        print(f"    P({class_name}|features): {test_proba[i, class_idx]:.4f}")
    status = "✓" if true == pred else "✗"
    print(f"  {status}")

## Summary Statistics

In [ ]:
# Plot metrics
metrics = ['Train Accuracy', 'Val Accuracy', 'Test Accuracy']
scores = [train_accuracy, val_accuracy, test_accuracy]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy comparison
colors = ['green', 'blue', 'red']
bars = ax1.bar(metrics, scores, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Accuracy')
ax1.set_ylim([0, 1])
ax1.set_title('Accuracy Across Different Datasets')
ax1.grid(axis='y', alpha=0.3)

for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}', ha='center', va='bottom')

# Plot 2: Metrics
test_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
test_scores = [test_accuracy, precision, recall, f1]
bars = ax2.bar(test_metrics, test_scores, color='steelblue', alpha=0.7, edgecolor='black')
ax2.set_ylabel('Score')
ax2.set_ylim([0, 1])
ax2.set_title('Performance Metrics on Test Set')
ax2.grid(axis='y', alpha=0.3)

for bar, score in zip(bars, test_scores):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("Performance visualization complete.")

## Viva Questions

### Basic Concepts
1. **What is Naïve Bayes theorem and its formula?**
   - Formula: P(class|features) = P(features|class) * P(class) / P(features)
   - Calculates posterior probability of class given features
   - Uses Bayes' theorem from probability theory

2. **Why is it called "Naïve" Bayesian classifier?**
   - Assumes conditional independence of all features given class
   - This assumption is naive/unrealistic but simplifies computation
   - Rarely true in real-world data but works well in practice

3. **What is the conditional independence assumption?**
   - P(feature1, feature2|class) = P(feature1|class) * P(feature2|class)
   - Allows factoring of joint probability
   - Reduces computational complexity from exponential to linear

### Implementation Questions
4. **How does Naïve Bayes calculate class probability?**
   - P(class|features) ∝ P(class) * ∏P(feature_i|class)
   - Multiplies prior by likelihood of each feature
   - Selects class with highest probability

5. **What is prior probability in Naïve Bayes?**
   - P(class): Probability of class before seeing any features
   - Often estimated as proportion of class in training data
   - Used to bias classifier toward more frequent classes

6. **How are likelihood probabilities estimated?**
   - For Gaussian NB: assumes feature follows Gaussian distribution
   - Parameters (mean, variance) estimated from training data
   - P(feature|class) calculated using Gaussian PDF

### Practical Questions
7. **What are advantages of Naïve Bayes?**
   - Simple and fast to train
   - Works well with limited training data
   - Handles high-dimensional data efficiently
   - Good baseline classifier

8. **What are disadvantages of Naïve Bayes?**
   - Assumes conditional independence (rarely true)
   - Cannot capture feature interactions
   - Biased toward dominant classes
   - May struggle with imbalanced datasets

9. **When should you use Naïve Bayes?**
   - Text classification and spam detection
   - Quick baseline for binary classification
   - Real-time classification requirements
   - Limited training data scenarios

10. **How does Naïve Bayes differ from logistic regression?**
    - NB is generative (models P(X|Y)), LR is discriminative (models P(Y|X))
    - NB makes strong independence assumption, LR doesn't
    - NB typically needs less data than LR
    - LR generally more accurate on large datasets